In [1]:
!python3 -V
import pDESy
from IPython.display import Markdown
from pDESy.model.base_project import BaseProject
from pDESy.model.base_project import datetime
from pDESy.model.base_product import BaseProduct
from pDESy.model.base_component import BaseComponent
from pDESy.model.base_workflow import BaseWorkflow
from pDESy.model.base_task import BaseTask
from pDESy.model.base_task import BaseTaskDependency
from pDESy.model.base_team import BaseTeam
from pDESy.model.base_worker import BaseWorker
from pDESy.model.base_facility import BaseFacility
from pDESy.model.base_workplace import BaseWorkplace
from pDESy.model.base_priority_rule import TaskPriorityRuleMode,ResourcePriorityRuleMode

Python 3.13.5


In [2]:
pDESy.__version__
pDESy.__file__

'/Users/keisukehirukawa/dev/pDESy_v0.7.3/pDESy/__init__.py'

製品定義

In [3]:
project = BaseProject("sample_workflow")
project = BaseProject(init_datetime = datetime.datetime(2025, 1, 1, 0, 0, 0), unit_timedelta=datetime.timedelta(minutes=60))

product = project.create_product("product")

A = product.create_component("A")
A.insert_absence_time_list([1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0])
B = product.create_component("B")

ワークフロー定義・同一ワークフロー間依存関係

In [4]:
workflowA = project.create_workflow("workflowA")

task_A1 = workflowA.create_task("A1", default_work_amount=24.0)
task_A2 = workflowA.create_task("A2", default_work_amount=24.0)
task_A3 = workflowA.create_task("A3", default_work_amount=24.0)

A.update_targeted_task_set({task_A1, task_A2, task_A3})

task_A2.add_input_task(task_A1)
task_A3.add_input_task(task_A2)

In [5]:
workflowB = project.create_workflow("workflowB")

task_B1 = workflowB.create_task("B1", default_work_amount=24.0)
task_B2 = workflowB.create_task("B2", default_work_amount=24.0)
task_B3 = workflowB.create_task("B3", default_work_amount=24.0)

B.update_targeted_task_set({task_B1, task_B2, task_B3})

task_B2.add_input_task(task_B1)
task_B3.add_input_task(task_B2)

異なるワークフロー間依存関係

In [6]:
task_B1.add_input_task(task_A2)

設備・人員

In [7]:
# wrokplace model
placeA = project.create_workplace("placeA", max_space_size=10.0)
placeB = project.create_workplace("placeB", max_space_size=10.0)

facilityA = placeA.create_facility("facilityA", cost_per_time=1)
facilityA.insert_absence_time_list([1,2,3,4,5,6])
facilityB = placeB.create_facility("facilityB", cost_per_time=1)
facilityA.workamount_skill_mean_map = {task_A1.name:1.0,task_A2.name:1.0, task_A3.name:1.0} 
facilityB.workamount_skill_mean_map = {task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0} 

#team model
team = project.create_team("team1")

wA = team.create_worker("workerA", cost_per_time=10.0, work_constraint_list=[(24,8, "FIXED"), (168,40, "FIXED"), (24,14, "SLIDING"), (168,72, "SLIDING")], rest_constraint_list=[(24,6)])
wB = team.create_worker("workerB", cost_per_time=10.0, work_constraint_list=[(24,8, "FIXED"), (168,40, "FIXED"), (24,14, "SLIDING"), (168,72, "SLIDING")], rest_constraint_list=[(24,6)])

wA.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0, task_A3.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0,
    } 
wB.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0
    }

wA.facility_skill_map = {facilityA.name:1.0, facilityB.name:1.0}
wB.facility_skill_map = wA.facility_skill_map.copy()

team.update_targeted_task_set({task_A1,task_A2,task_A3, task_B1,task_B2,task_B3})

placeA.update_targeted_task_set({task_A1,task_A2,task_A3})
placeB.update_targeted_task_set({task_B1,task_B2,task_B3})

In [8]:
project.simulate(max_time=200, progress_bar=True)

Simulating:   0%|          | 0/200 [00:00<?, ?time/s]/Users/keisukehirukawa/dev/pDESy_v0.7.3/pDESy/model/base_project.py:719: UserWarning: Time Over! Please check your simulation model or increase max_time value
  warnings.warn(
Time Over: 100%|██████████| 200/200 [00:00<00:00, 29877.15time/s] 


In [9]:
workflowA.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [10]:
workflowB.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [11]:
from plotly.figure_factory import create_gantt

task_id_list = [
    task_A1.ID, task_A2.ID, task_A3.ID,
    task_B1.ID, task_B2.ID, task_B3.ID
]

all_workflows = [
    workflowA,
    workflowB
]

combined_df = []
for wf in all_workflows:
    combined_df.extend(
        wf.create_data_for_gantt_plotly(
            init_datetime=project.init_datetime,
            unit_timedelta=project.unit_timedelta,
            target_id_order_list=list(task_id_list),
            print_workflow_name=True,
            view_ready=False,           # READY も表示したいなら True
            finish_margin=1.0
        )
    )

colors = {"WORKING": "rgb(146, 237, 5)", "READY": "rgb(107,127,135)"}

fig = create_gantt(
    combined_df,
    title="All Workflows Gantt",
    colors=colors,
    index_col="State",
    showgrid_x=True,
    showgrid_y=True,
    group_tasks=True,
    show_colorbar=True,
)

fig.show()

In [12]:
team.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [13]:
diagram = "flowchart TB\n" + "\n".join(project.get_mermaid_diagram())
display(Markdown(f"```mermaid\n{diagram}\n```"))

```mermaid
flowchart TB
subgraph b04b937a-ebd7-4657-8a8c-6ee7190f9c0a[product]
direction LR
b694fe65-9d1b-4d40-ba59-8beb59286ad1@{shape: odd, label: 'B'}
e3a42e9b-7b9f-4fd3-87d7-ba4a8474258f@{shape: odd, label: 'A'}
end
subgraph 667a0325-878e-4c28-8ddd-9c7132e1bd19[workflowA]
direction LR
aa3d3bb4-08a9-4520-81e0-f8f72e0e6254@{shape: rect, label: 'A3<br>0.0'}
d39a8c20-5afb-40a9-9783-6091123cb906@{shape: rect, label: 'A1<br>0.0'}
6de236e2-93c5-4af9-912f-23f87f5f646d@{shape: rect, label: 'A2<br>0.0'}
6de236e2-93c5-4af9-912f-23f87f5f646d-->aa3d3bb4-08a9-4520-81e0-f8f72e0e6254
d39a8c20-5afb-40a9-9783-6091123cb906-->6de236e2-93c5-4af9-912f-23f87f5f646d
end
subgraph 941e75f1-5574-4f89-b545-6613753167dc[workflowB]
direction LR
be28fff2-2068-43f8-9bb6-70afbc803aae@{shape: rect, label: 'B1<br>0.0'}
26ecec71-3a66-4ced-b640-e64d89e144e3@{shape: rect, label: 'B2<br>8.0'}
0db8084e-c238-4d86-b647-37da658bc97c@{shape: rect, label: 'B3<br>24.0'}
be28fff2-2068-43f8-9bb6-70afbc803aae-->26ecec71-3a66-4ced-b640-e64d89e144e3
26ecec71-3a66-4ced-b640-e64d89e144e3-->0db8084e-c238-4d86-b647-37da658bc97c
end
subgraph d72324f3-db25-4cc6-b6d2-2316fdfac85f[team1]
direction LR
d0108f39-de72-43d4-9ff6-b7dc3d8f89f0@{shape: stadium, label: 'workerB'}
bb1a7396-9b02-43cd-a395-44ec8e1047d4@{shape: stadium, label: 'workerA'}
end
subgraph c739cf50-8a2a-43df-906b-afd4eb183466[placeA]
direction LR
19f1a9c6-8a25-4483-8433-d423293e2777@{shape: stadium, label: 'facilityA'}
end
subgraph 1f19b041-7745-490f-a1b7-ffb4cd724345[placeB]
direction LR
ba9e97fc-0c53-49ef-a179-813068c0a348@{shape: stadium, label: 'facilityB'}
end
b694fe65-9d1b-4d40-ba59-8beb59286ad1-.-be28fff2-2068-43f8-9bb6-70afbc803aae
b694fe65-9d1b-4d40-ba59-8beb59286ad1-.-26ecec71-3a66-4ced-b640-e64d89e144e3
b694fe65-9d1b-4d40-ba59-8beb59286ad1-.-0db8084e-c238-4d86-b647-37da658bc97c
e3a42e9b-7b9f-4fd3-87d7-ba4a8474258f-.-aa3d3bb4-08a9-4520-81e0-f8f72e0e6254
e3a42e9b-7b9f-4fd3-87d7-ba4a8474258f-.-d39a8c20-5afb-40a9-9783-6091123cb906
e3a42e9b-7b9f-4fd3-87d7-ba4a8474258f-.-6de236e2-93c5-4af9-912f-23f87f5f646d
aa3d3bb4-08a9-4520-81e0-f8f72e0e6254-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
c739cf50-8a2a-43df-906b-afd4eb183466-.-aa3d3bb4-08a9-4520-81e0-f8f72e0e6254
d39a8c20-5afb-40a9-9783-6091123cb906-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
c739cf50-8a2a-43df-906b-afd4eb183466-.-d39a8c20-5afb-40a9-9783-6091123cb906
6de236e2-93c5-4af9-912f-23f87f5f646d-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
c739cf50-8a2a-43df-906b-afd4eb183466-.-6de236e2-93c5-4af9-912f-23f87f5f646d
be28fff2-2068-43f8-9bb6-70afbc803aae-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
1f19b041-7745-490f-a1b7-ffb4cd724345-.-be28fff2-2068-43f8-9bb6-70afbc803aae
26ecec71-3a66-4ced-b640-e64d89e144e3-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
1f19b041-7745-490f-a1b7-ffb4cd724345-.-26ecec71-3a66-4ced-b640-e64d89e144e3
0db8084e-c238-4d86-b647-37da658bc97c-.-d72324f3-db25-4cc6-b6d2-2316fdfac85f
1f19b041-7745-490f-a1b7-ffb4cd724345-.-0db8084e-c238-4d86-b647-37da658bc97c
6de236e2-93c5-4af9-912f-23f87f5f646d-->be28fff2-2068-43f8-9bb6-70afbc803aae
```